# 01 - Feature selection

**First step of the ML pipeline.** Loads one prepared feature panel, then
runs a multi-method feature-selection consensus to produce a final
modelling shortlist.

Pipeline:

| Section | Method | Purpose |
|---------|--------|---------|
| A | Load + validate panel | verify schema, keys, checkpoint encoding |
| B | Inspect panel | distribution, NaN, leakage checks |
| C | Variance filter | drop near-constant features |
| D | Univariate signal | Pearson r + mutual information vs target |
| E | Multicollinearity | greedy pruning at \|rho\| >= 0.9 |
| F | Tree importance | LightGBM with grouped CV (cluster on fixture_id) |
| G | L1 logistic | LASSO-style sparse selection |
| H | Consensus shortlist | features surviving 2+ methods |

## Setup

In [1]:
"""Imports and project paths."""
from __future__ import annotations
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
"""Display + warning config."""
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.precision", 4)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [3]:
"""Paths + single panel load."""
DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = PROJECT_ROOT / "features"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

panel_path = FEATURES_DIR / "all_data.csv"
PATH_MAIN = DATA_DIR / "all_data.csv"

assert panel_path.exists(), f"missing {panel_path}"
assert PATH_MAIN.exists(), f"missing {PATH_MAIN}"

panel = pd.read_csv(panel_path)


## Section A - Load & validate panel

We use a single pre-built modelling panel (`features/all_data.csv`) keyed
by `(player_appearance_id, checkpoint)`. In this section we verify basic
shape, key fields, and checkpoint encoding consistency before feature
selection.

In [4]:
"""Single-panel sanity check."""
required_cols = [
    "player_appearance_id",
    "checkpoint",
    "fixture_id",
    "position",
    "scored_after",
]
missing = [c for c in required_cols if c not in panel.columns]
assert not missing, f"panel is missing required columns: {missing}"

print(f"panel shape: {panel.shape} ({panel.shape[1]} cols)")
print(f"panel checkpoints: {sorted(panel['checkpoint'].dropna().unique())}")

panel shape: (3114, 123) (123 cols)
panel checkpoints: ['H1_15', 'H1_30', 'H1_45', 'H2_15', 'H2_30']


In [5]:
# Count unique players by player_appearance_id
id_col = "player_appearance_id"

n_panel_pa = panel[id_col].nunique()
print(f"Unique player_appearance_id in panel: {n_panel_pa:,}")

Unique player_appearance_id in panel: 796


In [6]:
# `panel` is already loaded in Setup from `panel_path`.
print("panel is loaded and ready")

panel is loaded and ready


In [7]:
"""Normalize checkpoint codes to canonical period-local form."""
CHECKPOINT_REMAP = {
    "H2_60": "H2_15",
    "H2_75": "H2_30",
    "H2_90": "H2_45",
}

panel = panel.copy()
panel["checkpoint"] = panel["checkpoint"].replace(CHECKPOINT_REMAP)

assert panel["checkpoint"].notna().all(), "checkpoint contains NaN after remap"
print(f"after remap: {sorted(panel['checkpoint'].unique())}")
print(panel["checkpoint"].value_counts().to_string())

after remap: ['H1_15', 'H1_30', 'H1_45', 'H2_15', 'H2_30']
checkpoint
H1_15    637
H1_30    635
H1_45    614
H2_15    614
H2_30    614


### Coverage analysis

Compare key coverage of the modelling `panel` against the raw main table
to understand how many `(player_appearance_id, checkpoint)` keys are
retained in the prepared dataset.

In [8]:
"""Coverage diff: main vs prepared panel."""
main = pd.read_csv(PATH_MAIN)

panel_keys = set(zip(panel["player_appearance_id"], panel["checkpoint"]))
main_keys = set(zip(main["player_appearance_id"], main["checkpoint"]))

print(f"main keys:  {len(main_keys):,}")
print(f"panel keys: {len(panel_keys):,}  ({len(panel_keys)/len(main_keys):.1%})")
print(f"intersection (panel x main): {len(panel_keys & main_keys):,}")
print()

diff = main_keys - panel_keys
diff_df = pd.DataFrame(list(diff), columns=["player_appearance_id", "checkpoint"])
,
diff_df = diff_df.merge(
    main[["player_appearance_id", "position", "checkpoint"]],
    on=["player_appearance_id", "checkpoint"],
    how="left",
)

print(f"rows in main but absent from panel ({len(diff_df):,}):")
if len(diff_df) > 0:
    print("  by position:")
    print(diff_df["position"].value_counts(dropna=False).to_string())
    print("  by checkpoint:")
    print(diff_df["checkpoint"].value_counts(dropna=False).to_string())
else:
    print("  none")

main keys:  3,114
panel keys: 3,114  (100.0%)
intersection (panel x main): 3,114

rows in main but absent from panel (0):
  none


## Section B - Panel inspect

Panel is already prepared, so this section performs key integrity and basic
readiness checks before downstream feature selection.

In [9]:
"""Panel integrity checks."""
JOIN = ["player_appearance_id", "checkpoint"]

n_dup = int(panel.duplicated(JOIN).sum())
print(f"panel shape: {panel.shape}")
print(f"duplicate keys on {JOIN}: {n_dup:,}")
assert n_dup == 0, "panel contains duplicate (player_appearance_id, checkpoint) keys"

print(f"rows with missing target: {panel['scored_after'].isna().sum():,}")

panel shape: (3114, 123)
duplicate keys on ['player_appearance_id', 'checkpoint']: 0
rows with missing target: 0


In [10]:
"""Target distribution."""
y_col = "scored_after"
n_pos = panel[y_col].sum()
print(f"target rate: {panel[y_col].mean():.4f} ({n_pos:,} / {len(panel):,})")
print()
print("target by position:")
print(panel.groupby("position")[y_col].agg(["size", "sum", "mean"]).round(4).to_string())
print()
print("target by checkpoint:")
print(panel.groupby("checkpoint")[y_col].agg(["size", "sum", "mean"]).round(4).to_string())


target rate: 0.0649 (202.0 / 3,114)

target by position:
          size   sum    mean
position                    
A          609  77.0  0.1264
D         1218  38.0  0.0312
M         1287  87.0  0.0676

target by checkpoint:
            size   sum    mean
checkpoint                    
H1_15        637  35.0  0.0549
H1_30        635  42.0  0.0661
H1_45        614  37.0  0.0603
H2_15        614  50.0  0.0814
H2_30        614  38.0  0.0619


In [11]:
"""Missingness audit on numeric columns only."""
numeric_panel = panel.select_dtypes(include="number")
na = numeric_panel.isna().sum()
na = na[na > 0].sort_values(ascending=False)
print(f"columns with NaN in numeric subset: {len(na)} / {len(numeric_panel.columns)}")
print()
print(na.head(20).to_string())


columns with NaN in numeric subset: 51 / 116

ratio_shots_on_target                      2663
ratio_shots_under_press                    2275
top_third_press_share_x_set_piece_share    2235
dominant_foot_strength                     2229
set_piece_share                            2229
top_third_run_share_x_set_piece_share      2229
share_right_foot                           2229
top_third_run_share_x_shot_accuracy        2229
shot_accuracy                              2229
share_left_foot                            2229
ratio_shots_top_third                      2131
ratio_shots                                2121
shots_per_top_third_run                     488
forward_pass_share                          445
mean_abs_pass_angle                         445
top_third_presence_joint                    275
composure_under_pressure_ratio              247
player_shot_share                           242
top_third_press_share                       222
shot_to_press_ratio                       

In [12]:
"""Identify candidate feature columns (drop IDs / target / categoricals
we will encode separately).
"""
ID_COLS = ["player_appearance_id", "player_id", "fixture_id", "date",
           "checkpoint", "checkpoint_period", "checkpoint_min"]
TARGET = "scored_after"
CAT_COLS = ["position", "formation", "is_home", "subbed"]

feature_cols = [c for c in panel.columns
                if c not in ID_COLS + CAT_COLS + [TARGET]
                and panel[c].dtype != "object"]
print(f"candidate numeric feature columns: {len(feature_cols)}")
print()
print(feature_cols)


candidate numeric feature columns: 111

['match_minute_at_cp', 'shot_accuracy', 'share_left_foot', 'share_right_foot', 'dominant_foot_strength', 'set_piece_share', 'is_penalty_taker', 'last15_intensity', 'cumul_intensity', 'intensity_uplift', 'player_shot_share', 'top_third_run_share', 'top_third_last15_share', 'runs_per_minute_played', 'cumul_press_events', 'last15_press_events', 'press_turnover_rate', 'top_third_press_share', 'forward_pass_share', 'mean_abs_pass_angle', 'cumul_pressing_others', 'pressing_minus_pressed', 'cumul_passes', 'last15_passes', 'top_third_pass_share', 'passes_received_share', 'shots_per_top_third_run', 'top_third_run_share_A', 'top_third_run_share_M', 'top_third_run_share_D', 'top_third_run_share_G', 'last15_sprints_A', 'last15_sprints_M', 'last15_sprints_D', 'last15_sprints_G', 'fatigue_gap', 'top_third_run_share_x_set_piece_share', 'top_third_run_share_x_shot_accuracy', 'shot_intent_under_fatigue', 'speed_above_team_avg', 'top_third_presence_joint', 'top_th

## Section C - Variance filter

Drop features that are constant or near-constant (variance below 1e-8 or
more than 99% of values identical). These carry no information for any
model.


In [13]:
"""Variance & near-constant filter."""
def is_near_constant(series: pd.Series, threshold: float = 0.99) -> bool:
    if series.dropna().nunique() <= 1:
        return True
    top_share = series.value_counts(normalize=True, dropna=False).iloc[0]
    return top_share >= threshold

near_const = [c for c in feature_cols if is_near_constant(panel[c])]
zero_var = [c for c in feature_cols
            if panel[c].dropna().nunique() <= 1]

print(f"near-constant (>=99% same value): {len(near_const)}")
for c in near_const:
    print(f"  {c}")

feature_cols_post_var = [c for c in feature_cols if c not in near_const]
print(f"\nafter variance filter: {len(feature_cols_post_var)} / {len(feature_cols)}")


near-constant (>=99% same value): 2
  is_penalty_taker
  last15_sprints_G

after variance filter: 109 / 111


## Section D - Univariate signal

Two complementary measures of association with the target:

* **Pearson r** - linear / monotone signal (point-biserial when one side
  is binary).
* **Mutual information** - non-parametric, captures non-monotone
  relationships that Pearson misses.

We compute both for the post-variance feature set, rank by absolute
strength, and use the top-half as the "univariate-strong" subset for
later consensus.


In [14]:
"""Pearson r vs target."""
y = panel[TARGET]
r_rows = []
for c in feature_cols_post_var:
    s = panel[c].dropna()
    if s.std() == 0:
        continue
    t = y.loc[s.index]
    r = float(np.corrcoef(s, t)[0, 1])
    r_rows.append({"feature": c, "n": len(s), "pearson_r": r})
r_df = pd.DataFrame(r_rows).sort_values("pearson_r", key=lambda x: x.abs(), ascending=False)
r_df.reset_index(drop=True).head(25)


,feature,n,pearson_r
0,minutes_remaining,3114,0.1016
1,ratio_peak_speed,3114,-0.1002
2,ratio_mean_max_speed,3114,-0.0995
3,minutes_active,3114,-0.0986
4,active_windows,3114,-0.0986
5,cumul_hsr,3114,-0.0729
6,jersey_number,3114,0.0659
7,last15_shots_top_third,3114,0.0531
8,formation_offensiveness,3114,-0.0527
9,cumul_sprints,3114,-0.0524


In [15]:
"""Mutual information."""
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler

# MI handles NaN poorly - impute with 0 (matches our downstream policy
# of treating NaN ratios as "no history yet").
X_imputed = panel[feature_cols_post_var].fillna(0.0)

mi = mutual_info_classif(X_imputed, y, random_state=42, discrete_features=False)
mi_df = (pd.DataFrame({"feature": feature_cols_post_var, "mi": mi})
         .sort_values("mi", ascending=False)
         .reset_index(drop=True))
mi_df.head(25)


,feature,mi
0,cumul_peak_speed,0.0310
1,last15_passes_top_third,0.0249
2,minute_out,0.0169
3,jersey_number,0.0133
4,ratio_sprints,0.0104
5,cumul_sprints,0.0090
6,top_third_presence_joint,0.0077
7,ratio_mean_max_speed,0.0074
8,active_windows,0.0074
9,cumul_passes_A,0.0073


In [16]:
"""Combine both univariate measures."""
univ = r_df.merge(mi_df, on="feature", how="outer")
univ["abs_r"] = univ["pearson_r"].abs()
univ = univ.sort_values("abs_r", ascending=False).reset_index(drop=True)
univ.head(30)


,feature,n,pearson_r,mi,abs_r
0,minutes_remaining,3114,0.1016,3.3838e-05,0.1016
1,ratio_peak_speed,3114,-0.1002,2.3516e-03,0.1002
2,ratio_mean_max_speed,3114,-0.0995,7.4424e-03,0.0995
3,minutes_active,3114,-0.0986,1.7297e-03,0.0986
4,active_windows,3114,-0.0986,7.3584e-03,0.0986
5,cumul_hsr,3114,-0.0729,0.0000e+00,0.0729
6,jersey_number,3114,0.0659,1.3338e-02,0.0659
7,last15_shots_top_third,3114,0.0531,3.3617e-03,0.0531
8,formation_offensiveness,3114,-0.0527,0.0000e+00,0.0527
9,cumul_sprints,3114,-0.0524,8.9818e-03,0.0524


## Section E - Multicollinearity pruning

Greedy pruning by descending |Pearson|: walk down the ranked list, keep a
feature only if its |rho| with every already-kept feature is below
0.90. This removes the worst redundancies before tree / L1 selection
without sacrificing top signals.


In [17]:
"""Greedy correlation prune."""
RHO_THRESHOLD = 0.90

X = panel[feature_cols_post_var].fillna(0.0)
corr = X.corr().abs()
ordered = univ.sort_values("abs_r", ascending=False)["feature"].tolist()

kept = []
dropped = []
for f in ordered:
    if f not in corr.columns:
        continue
    conflict = None
    for k in kept:
        rho = corr.at[f, k]
        if pd.notna(rho) and rho >= RHO_THRESHOLD:
            conflict = (k, float(rho))
            break
    if conflict is None:
        kept.append(f)
    else:
        dropped.append((f, conflict[0], conflict[1]))

print(f"kept after collinearity prune: {len(kept)}")
print(f"dropped: {len(dropped)}")
print()
if dropped[:10]:
    print("dropped (feature, partner, rho):")
    for d, k, rho in dropped[:10]:
        print(f"  {d:55s} <- {k:40s} rho={rho:.3f}")

feature_cols_post_collin = kept


kept after collinearity prune: 97
dropped: 12

dropped (feature, partner, rho):
  ratio_mean_max_speed                                    <- ratio_peak_speed                         rho=0.985
  minutes_active                                          <- ratio_peak_speed                         rho=0.977
  active_windows                                          <- ratio_peak_speed                         rho=0.977
  last15_shots                                            <- last15_shots_top_third                   rho=0.984
  cumul_shots_top_third                                   <- cumul_shots                              rho=0.992
  shot_accuracy                                           <- top_third_run_share_x_shot_accuracy      rho=0.907
  last15_shots_on_target                                  <- last15_shot_accuracy                     rho=0.944
  top_third_press_share                                   <- top_third_presence_avg                   rho=0.909
  cumul_shots_under_pres

## Section F - Tree importance (LightGBM with grouped CV)

The most reliable single signal for tabular classification: tree
splitting gain aggregated across folds. Group K-fold on `fixture_id`
prevents within-match leakage (multiple players per match). We average
gain across folds rather than using a single training pass.


In [18]:
"""LightGBM with GroupKFold(fixture_id)."""
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

X = panel[feature_cols_post_collin].fillna(0.0).values
y_arr = panel[TARGET].values
groups = panel["fixture_id"].values

gkf = GroupKFold(n_splits=5)
gain_per_fold = []
for fold, (tr, va) in enumerate(gkf.split(X, y_arr, groups)):
    train = lgb.Dataset(X[tr], label=y_arr[tr], feature_name=feature_cols_post_collin)
    valid = lgb.Dataset(X[va], label=y_arr[va], feature_name=feature_cols_post_collin)
    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.03,
        "num_leaves": 31,
        "feature_fraction": 0.85,
        "bagging_fraction": 0.85,
        "bagging_freq": 5,
        "min_child_samples": 20,
        "verbosity": -1,
        "seed": 42 + fold,
    }
    booster = lgb.train(
        params, train,
        num_boost_round=300,
        valid_sets=[valid],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)],
    )
    g = booster.feature_importance(importance_type="gain")
    gain_per_fold.append(pd.Series(g, index=feature_cols_post_collin))
    print(f"fold {fold+1}: best AUC={booster.best_score['valid_0']['auc']:.4f}, "
          f"best_iter={booster.best_iteration}")

importance = pd.concat(gain_per_fold, axis=1).mean(axis=1).sort_values(ascending=False)
imp_df = importance.reset_index().rename(columns={"index": "feature", 0: "gain_mean"})
imp_df.head(30)


fold 1: best AUC=0.6531, best_iter=2
fold 2: best AUC=0.7345, best_iter=51
fold 3: best AUC=0.5252, best_iter=1
fold 4: best AUC=0.6268, best_iter=4
fold 5: best AUC=0.6890, best_iter=83


,feature,gain_mean
0,jersey_number,558.9236
1,last15_passes_top_third,403.2383
2,cumul_peak_speed,339.1873
3,minutes_remaining,259.5498
4,minute_out,189.3214
5,last15_distance,173.4864
6,speed_above_team_avg,170.6251
7,formation_offensiveness,166.4544
8,cumul_mean_max_speed,152.4528
9,cumul_distance,136.9858


## Section G - L1-regularised logistic regression

LASSO-style sparse feature selection: features with non-zero coefficients
at a moderate regularisation strength are deemed informative under a
linear model. Standard-scaled features required.


In [19]:
"""L1 logistic with standard scaling + cross-validated C."""
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X = panel[feature_cols_post_collin].fillna(0.0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_arr = panel[TARGET].values

# Single fit at moderate L1 strength (C=0.5). CV-tuned C is also fine
# but for a feature-selection sweep one moderate point is sufficient.
clf = LogisticRegression(
    penalty="l1", solver="liblinear", C=0.5,
    class_weight="balanced", max_iter=2000, random_state=42,
)
clf.fit(X_scaled, y_arr)

l1_df = pd.DataFrame({
    "feature": feature_cols_post_collin,
    "coef": clf.coef_[0],
    "abs_coef": np.abs(clf.coef_[0]),
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)

n_nonzero = (l1_df["abs_coef"] > 1e-8).sum()
print(f"non-zero L1 coefficients: {n_nonzero} / {len(l1_df)}")
l1_df.head(30)


non-zero L1 coefficients: 82 / 97


,feature,coef,abs_coef
0,ratio_peak_speed,-0.5083,0.5083
1,minutes_remaining,0.3368,0.3368
2,jersey_number,0.2964,0.2964
3,top_third_pass_x_top_third_run,-0.2938,0.2938
4,last15_sprints_A,-0.2846,0.2846
5,last15_passes,-0.2712,0.2712
6,formation_offensiveness,-0.2648,0.2648
7,ratio_shots_top_third,0.2612,0.2612
8,top_third_press_share_above_team_avg,0.2581,0.2581
9,runs_per_minute_played,0.2552,0.2552


## Section H - Consensus shortlist

Combine the four ranked lists (Pearson r, MI, tree gain, L1 |coef|) into
a single consensus score: a feature gets one "vote" per method where it
ranks in the top-K, then features are sorted by total votes (ties broken
by mean rank). Features surviving 3 out of 4 methods form the high-
confidence shortlist; 2-of-4 forms the soft shortlist.


In [20]:
"""Consensus voting."""
TOP_K = 25

ranks = pd.DataFrame({"feature": feature_cols_post_collin})

# Pearson rank
pr = univ.set_index("feature")["abs_r"]
ranks = ranks.merge(pr.rank(ascending=False).rename("rank_pearson"),
                    left_on="feature", right_index=True, how="left")
# MI rank
mr = univ.set_index("feature")["mi"]
ranks = ranks.merge(mr.rank(ascending=False).rename("rank_mi"),
                    left_on="feature", right_index=True, how="left")
# Tree gain rank
ranks = ranks.merge(importance.rank(ascending=False).rename("rank_tree"),
                    left_on="feature", right_index=True, how="left")
# L1 |coef| rank
lr = l1_df.set_index("feature")["abs_coef"]
ranks = ranks.merge(lr.rank(ascending=False).rename("rank_l1"),
                    left_on="feature", right_index=True, how="left")

# Votes: 1 per method where feature is in top-K
for col in ("rank_pearson", "rank_mi", "rank_tree", "rank_l1"):
    ranks[f"top{TOP_K}_{col.split('_')[1]}"] = (ranks[col] <= TOP_K).astype(int)

ranks["votes"] = ranks[[c for c in ranks.columns if c.startswith(f"top{TOP_K}_")]].sum(axis=1)
ranks["mean_rank"] = ranks[["rank_pearson", "rank_mi", "rank_tree", "rank_l1"]].mean(axis=1)
ranks = ranks.sort_values(["votes", "mean_rank"], ascending=[False, True]).reset_index(drop=True)

print("vote distribution (out of 4 methods):")
print(ranks["votes"].value_counts().sort_index(ascending=False).to_string())
print()
print(f"top {TOP_K + 5} features by consensus:")
ranks.head(TOP_K + 5)


vote distribution (out of 4 methods):
votes
4     1
3     9
2    17
1    24
0    46

top 30 features by consensus:


,feature,rank_pearson,rank_mi,rank_tree,rank_l1,top25_pearson,top25_mi,top25_tree,top25_l1,votes,mean_rank
0,jersey_number,7.0,4.0,1.0,3.0,1,1,1,1,4,3.750
1,ratio_peak_speed,2.0,34.0,13.0,1.0,1,0,1,1,3,12.500
2,minutes_remaining,1.0,53.0,4.0,2.0,1,0,1,1,3,15.000
3,last15_passes_top_third,12.0,2.0,2.0,46.0,1,1,1,0,3,15.500
4,runs_per_minute_played,28.0,15.0,19.0,10.0,0,1,1,1,3,18.000
5,cumul_sprints,10.0,6.0,24.0,48.0,1,1,1,0,3,22.000
6,minute_out,24.0,3.0,5.0,58.0,1,1,1,0,3,22.500
7,formation_offensiveness,9.0,81.5,8.0,7.0,1,0,1,1,3,26.375
8,ratio_passes_top_third,20.0,21.0,46.0,24.0,1,1,0,1,3,27.750
9,top_third_press_share_above_team_avg,15.0,81.5,12.0,9.0,1,0,1,1,3,29.375


In [21]:
"""Final shortlist."""
shortlist_high = ranks[ranks["votes"] >= 3]["feature"].tolist()
shortlist_soft = ranks[ranks["votes"] >= 2]["feature"].tolist()
print(f"high-confidence shortlist (>=3/4 methods): {len(shortlist_high)} features")
for f in shortlist_high:
    print(f"  {f}")
print()
print(f"soft shortlist (>=2/4 methods): {len(shortlist_soft)} features")


high-confidence shortlist (>=3/4 methods): 10 features
  jersey_number
  ratio_peak_speed
  minutes_remaining
  last15_passes_top_third
  runs_per_minute_played
  cumul_sprints
  minute_out
  formation_offensiveness
  ratio_passes_top_third
  top_third_press_share_above_team_avg

soft shortlist (>=2/4 methods): 27 features


## Persist outputs

Three artefacts saved for the next pipeline step:

* `models/feature_selection_panel.csv` - full merged panel.
* `models/feature_importance.csv` - importance scores by method.
* `models/feature_selection_shortlist.csv` - the high-confidence shortlist.


In [23]:
"""Persist."""
panel.to_csv(MODELS_DIR / "feature_selection_panel.csv", index=False)

importance_export = ranks[
    ["feature", "rank_pearson", "rank_mi", "rank_tree", "rank_l1",
     "votes", "mean_rank"]
].copy()
importance_export = importance_export.merge(
    univ[["feature", "pearson_r", "mi"]], on="feature", how="left",
).merge(
    importance.reset_index().rename(columns={"index": "feature", 0: "tree_gain_mean"}),
    on="feature", how="left",
).merge(
    l1_df[["feature", "coef", "abs_coef"]].rename(columns={"coef": "l1_coef", "abs_coef": "l1_abs_coef"}),
    on="feature", how="left",
)
importance_export.to_csv(MODELS_DIR / "feature_importance.csv", index=False)

shortlist_df = (
    panel[["player_appearance_id", "checkpoint", TARGET, "fixture_id",
           "is_home", "position"] + shortlist_high]
    .copy()
)
shortlist_df.to_csv(MODELS_DIR / "feature_selection_shortlist.csv", index=False)

print(f"wrote: models/feature_selection_panel.csv      ({panel.shape})")
print(f"wrote: models/feature_importance.csv           ({importance_export.shape})")
print(f"wrote: models/feature_selection_shortlist.csv  ({shortlist_df.shape})")


wrote: models/feature_selection_panel.csv      ((3114, 123))
wrote: models/feature_importance.csv           ((97, 12))
wrote: models/feature_selection_shortlist.csv  ((3114, 16))


### Next step

The shortlist CSV is keyed on `(player_appearance_id, checkpoint)` and
includes the target, group key, and 6 base context columns + N selected
features. The next notebook (`02_baseline_model.ipynb`) will:

1. Set up the modelling table from this shortlist.
2. Define grouped-K-fold cross-validation (cluster on `fixture_id`).
3. Fit baseline LightGBM + logistic-regression models.
4. Calibrate probabilities and threshold-optimise for balanced accuracy.
5. Report AUC + balanced accuracy on the test fold + overall.
